# SCENIC+ transcription factor perturbation

For this tutorial, we are processing the medulloblastoma mouse model single cell multiome data from:

[_Shiraishi, R. & Cancila, G. et al. (2024). Cancer-specific epigenome identifies oncogenic hijacking by nuclear factor I family proteins for medulloblastoma progression. Dev. Cell , 59:2302-2319._](https://www.cell.com/developmental-cell/fulltext/S1534-5807(24)00330-7)

This data set contains three samples comprising FACS sorted cells. These cells reflect the progression from healthy precursors to tumor cells:

1. Ptch1GNP: granule neuron precursors, P7
2. PNC: preneoplastic cells, P28
3. Tumor: tumor cells, adult mice

Data was downloaded from the Gene Expression Omnibus:

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE240362

The `GSE240362_RAW.tar` file contains the raw gene expression counts files in HDF5 format (.h5) and ATAC fragment files. This is the typical output you would obtain from the [10X Genomics Cellranger software](https://www.10xgenomics.com/support/software/cell-ranger/latest/getting-started/cr-what-is-cell-ranger).

The ATAC modality (fragment files) were processed with _pycisTopic_ and gene expression (.5) was processed with _Scanpy_. In addition, _cisTarget_ transcription factor binding site motif databases were generated. The SCENIC+ pipelines was then used to infer eRegulon. To see how the processing was done, see the following notebooks:

* [scanpy_rna_processing.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/scanpy_rna_processing.ipynp)
* [pycistopic_atac_processing.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/pycistopic_atac_processing.ipynp)
* [pycistarget_tfmotif_database.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/pycistarget_tfmotif_database.ipynp)
* [scenicplus_pipeline.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/scenicplus_pipeline.ipynp)

For more details on SCENIC+, have a look at:

* [_Bravo González-Blas, C., De Winter, S., Hulselmans, G., Hecker, N., Matetovici, I., Christiaens, V., ... & Aerts, S. (2023). SCENIC+: single-cell multiomic inference of enhancers and gene regulatory networks. Nat. Methods, 20:1355-1367._](https://www.nature.com/articles/s41592-023-01938-4)
* [SCENIC+ documentation](https://scenicplus.readthedocs.io/en/latest/)

It is assumed that you are already familar with the SCENIC+ MuData data object that we inspected in the SCENIC+ analysis tutorial:
* [scenicplus_analysis.ipynb](https://github.com/heckern/2026_ebi_workshop_scenicplus/scenicplus_analysis.ipynb)


SCENIC+ uses gradient boosting machine (GBM) regression to predict the impact of transcription factor perturbation on target genes. This is done in two steps:

1. GBM regressors are trained which predict the expression of target genes based on the expression of transcription factors (TFs)
2. A perturbation is applied to a TF and the GBMs are used to predict after the change of the TF expression. The resulting gene expression matrix is then used as input for another iteration for prediction the gene expression. Typically, we perform around five iterations of updating the gene expression.

**A note of caution**: predicting the effect of TF perturbations can reveal interesting tendencies but should be taken with a grain of salt and be experimentally validated . The accuracy of the predictions remains limited and may further porpagate errors of potentially wrongfully inferred TF-target gene interactions by SCENIC+.  

To start, we load the SCENIC+ MuData object and required packages and fuctions.

In [ ]:
import mudata
import os
import scanpy as sc
import anndata
import matplotlib
import matplotlib.pyplot as plt
import adjustText
import numpy as np
import pandas as pd

from scenicplus.simulation import (
    train_gene_expression_models,
    simulate_perturbation,
    plot_perturbation_effect_in_embedding
)

In [ ]:
%matplotlib inline

In [ ]:
path_scplus = '/home/training/course_dir/data_dir/nhecker/scenicplus_mm10/out/scplusmdata.h5mu'

scplus_mdata = mudata.read(path_scplus)

In [ ]:
scplus_mdata

## Training of regression models

As a prepartion for training the GBMs, we create gene-to-TFs dictionary based on the inferred eRegulons.

In [ ]:
gene_tf_direct_extended = pd.concat(
    [
        scplus_mdata.uns["direct_e_regulon_metadata"][["Gene", "TF"]].drop_duplicates(),
        scplus_mdata.uns["extended_e_regulon_metadata"][["Gene", "TF"]].drop_duplicates()
    ]
).drop_duplicates()
gene_to_TF = gene_tf_direct_extended.groupby("Gene")["TF"].apply(lambda tfs: list(tfs)).to_dict()

For example, the gene _Gucy1a2_ is predicted to be regulated by _Nfia_, _Nfib_, and _Zbtb20_.

In [ ]:
gene_to_TF['Gucy1a2']

Since the TF perturbation simulation and GBM training can be a very resource intensiv process, we are limiting the analysis in this tutorial to the top 300 eRegulon interactions with direct motif annotations based on their `triplet_rank`.

In [ ]:
# use a subset of genes, just so the notebook runs fast
ntop = 300

eregulon_interactions = scplus_mdata.uns["direct_e_regulon_metadata"].sort_values("triplet_rank").iloc[0:ntop]
genes_to_use = eregulon_interactions["Gene"].drop_duplicates()

The top 300 interactions include 200 different genes.

In [ ]:
len(genes_to_use)

Training the GBMs for this dataset can take up to 15 minutes. So, you ideally spent your time with other investigation or preparations while waiting for the regressor training to finish.

In [ ]:
regressors = train_gene_expression_models(
    df_EXP = scplus_mdata["scRNA_counts"].to_df(),
    gene_to_TF = gene_to_TF,
    genes = genes_to_use,
)

## Simulation of TF pertubation

In this example, we are simulating a knockdown of Nfia by simply setting the expression of Nfia to 0: `{"Nfia": 0}`. We are performing 5 iterations (`n_iter = 5`) of updating the gene expression after this perturbation. This will take around 5 minutes in total.

In [ ]:
perturbation_over_iter = simulate_perturbation(
    df_EXP = scplus_mdata["scRNA_counts"].to_df(),
    perturbation = {"Nfia": 0},
    keep_intermediate = True,
    n_iter = 5,
    regressors = regressors
)

The simulated gene expression for the five iterations is stored in form of a list of gene expression matrices `perturbation_over_iter`.

To investigate the impact of predicted the Nfia knockdown, we look at the log-fold change of its top target genes after the simulated perturbation.

In [ ]:
eregulon_name = 'Nfia_direct_+/+'

scplus_mdata.uns['direct_e_regulon_metadata'][ scplus_mdata.uns['direct_e_regulon_metadata']['eRegulon_name'] == eregulon_name].sort_values('triplet_rank')[0:5]

We are picking the top 4 genes here.

In [ ]:
genes_to_show = ["Gucy1a2", "Robo2", "Ptprd", "Neo1"]

Here we are looking at the average predicted log-fold change of the genes for cells of the tumor samples.

In [ ]:
import scanpy as sc
import anndata

eRegulon_gene_AUC = anndata.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs

In [ ]:
sample = 'Tumor'

fig, ax = plt.subplots()

baseline = perturbation_over_iter[0][genes_to_show].groupby(eRegulon_gene_AUC.obs["scRNA_counts:sample"]).mean().loc[sample]

for gene in genes_to_show:
    ax.plot(
        np.arange(5)+1,
        [
            np.log2(perturbation_over_iter[i][genes_to_show].groupby(eRegulon_gene_AUC.obs["scRNA_counts:sample"]).mean().loc[sample] / baseline)[gene]
            for i in np.arange(5)+1
        ],
        label = gene
    )
ax.set_ylabel("Predicted $log{_2}FC$")
ax.set_xlabel("Iteration")
ax.legend()
ax.axhline(y = 0, color = "black")
ax.grid("gray")
ax.set_axisbelow(True)

You can view the initial mean expression values in the following way:

In [ ]:
perturbation_over_iter[0][genes_to_show].groupby(eRegulon_gene_AUC.obs["scRNA_counts:sample"]).mean()

And for the 5th iteration:

In [ ]:
perturbation_over_iter[5][genes_to_show].groupby(eRegulon_gene_AUC.obs["scRNA_counts:sample"]).mean()

## Visualisation of cell trajectories after perturbation

SCENIC+ allows us you to visualise trajectories which describe how cells would change after perturbation. In other words, how far and which direction would a cell move in a UMAP after perturbation of the gene expression. This is based on the [velocyto](https://velocyto.org/velocyto.py/index.html) framework. These trajectories can be interesting for understanding relationships between cells and whether certains TFs may function as cell-state switches.

First, we compute a UMAP based on the gene-based eRegulon AUC values.

In [ ]:
sc.pp.neighbors(eRegulon_gene_AUC, use_rep = "X")
sc.tl.umap(eRegulon_gene_AUC)

Next, we create a plotting a UMAP. This will allow us to plot cell trajectories on top of a UMAP. We are also specifying some cells for the three samples.

In [ ]:
color_dict = {
    'Ptch1GNP': 'blue',
    'PNC': 'orange',
    'Tumor': 'red',
}

In [ ]:
def plot_embedding(ax, embedding='X_umap', dot_size=1):
    texts = []

    ax.scatter(
        eRegulon_gene_AUC.obsm[embedding][:, 0],
        eRegulon_gene_AUC.obsm[embedding][:, 1],
        color = [color_dict[sample] for sample in eRegulon_gene_AUC.obs["scRNA_counts:sample"]],
        s = dot_size
    )
    # Plot labels
    for line in set(eRegulon_gene_AUC.obs["scRNA_counts:sample"]):
        line_bc_idc = np.arange(len(eRegulon_gene_AUC.obs_names))[eRegulon_gene_AUC.obs["scRNA_counts:sample"] == line]
        avg_x, avg_y = eRegulon_gene_AUC.obsm[embedding][line_bc_idc, 0:2].mean(0)
        texts.append(
            ax.text(
                avg_x,
                avg_y,
                line,
                fontweight = "bold"
            )
        )
    adjustText.adjust_text(texts)

fig, ax = plt.subplots()
plot_embedding(ax)

To lessen the computationl burden, we are subsetting the expression matrices to the genes from the top N eRegulation interactions as specified above.

In [ ]:
expression_matrix_initial = perturbation_over_iter[0][genes_to_use].copy()
expression_matrix_perturbed = perturbation_over_iter[5][genes_to_use].copy()

We first plot the UMAP. Next, we call the `plot_perturbation_effect_in_embedding` function to plot cell trajectories on top of the UMAP. This process will probably take around 5 minutes.

In [ ]:
fig, ax = plt.subplots()
plot_embedding(ax, 'X_umap')
plot_perturbation_effect_in_embedding(
    perturbed_matrix = expression_matrix_perturbed,
    original_matrix = expression_matrix_initial,
    embedding = eRegulon_gene_AUC.obsm["X_umap"][:, 0:2],
    AUC_kwargs = {},
    ax = ax,
    eRegulons = eregulon_interactions,
    n_cpu = 16
)

The color shade  of the arrows indicates the distance of the shift after simulated perturbation.